<a href="https://colab.research.google.com/github/igMoreira/claude-cert-notebooks/blob/main/build_with_claude_api/class_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
pip install Anthropic

In [54]:
from anthropic import Anthropic
from google.colab import userdata

MODEL = 'claude-haiku-4-5-20251001'
#MODEL = 'claude-sonnet-5'
MAX_TOKENS = 1000
API_KEY = userdata.get('API_KEY')
client = Anthropic(api_key=API_KEY)


In [63]:
import json

def generate_dataset():
    return """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

def user_message(messages, text):
  messages.append({'role': 'user', 'content': text})
  return messages

def assistant_message(messages, text):
  messages.append({'role': 'assistant', 'content': text})
  return messages

def chat(messages, system=None, stop_sequences=None):
  params = {
      'model':MODEL,
      'max_tokens':MAX_TOKENS,
      'messages':messages
  }
  if system:
    params['system'] = system
  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  response = client.messages.create(**params)
  answer = response.content[0].text
  assistant_message(messages, answer)
  return answer


messages = []

user_message(messages, generate_dataset())
assistant_message(messages, "```json")
answer = chat(messages, stop_sequences=["```"])

with open('dataset.json', 'w') as f:
  f.write(answer)
print(answer)


[
  {
    "task": "Write a Python function that takes an AWS S3 bucket name and returns True if it follows AWS naming conventions (lowercase, 3-63 characters, no underscores), False otherwise."
  },
  {
    "task": "Create a JSON object that represents an AWS IAM policy statement allowing read-only access to all objects in an S3 bucket named 'my-data-bucket'."
  },
  {
    "task": "Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters)."
  }
]

